# Providers 04 - Ollama Runtime

Objetivo: ejecutar un modelo local cuantizado mediante `ollama-runtime` y la
misma fachada pública que OpenAI, Bedrock y vLLM.

**Lugar en el modelo:** Ollama es el Provider local; el Agent y su Framework no
cambian. Agentic Systems no instala Ollama, no descarga modelos y no administra
el proceso del servidor.

**Evidencia exigida:** una prueba live produce `RunResult.ok=True` y
`engine="ollama-runtime"`. Un preflight o un contrato offline no demuestra que
el modelo respondió.

**Límite de la evidencia:** `not-run` sólo demuestra que el tutorial conserva
su contrato sin infraestructura; no certifica servidor, GPU, modelo ni tool
calling.


## Parametros de la demostracion

| Variable | Default | Propósito |
|---|---|---|
| RUN_OLLAMA_LIVE | 1 | Usa 0 para desactivar la llamada real. |
| OLLAMA_MODEL | qwen3:4b-instruct-2507-q4_K_M | Modelo Instruct 2507 reproducible para agentes y tool use. |
| OLLAMA_BASE_URL | http://127.0.0.1:11434/v1 | Endpoint OpenAI-compatible. |
| OLLAMA_API_KEY | ollama | Valor local convencional; nunca se imprime. |

Para una GPU de 8 GB empieza con un modelo pequeño o cuantizado. La elección
exacta es operativa y puede cambiar sin modificar la API del Agent.


## Contrato de la demostración

```text
toolkit.runtime -> toolkit.system -> system.agent -> RunResult
```

La ruta es idéntica a otros providers. Sólo cambian configuración, modelo y
evidencia operativa.


In [ ]:
import os

import agentic_systems as toolkit

ollama_environment = toolkit.ollama_environment_snapshot()

RUN_OLLAMA_LIVE = os.getenv("RUN_OLLAMA_LIVE", "1").strip().lower() in {
    "1", "true", "yes"
}
MODEL = os.getenv("OLLAMA_MODEL") or "qwen3:4b-instruct-2507-q4_K_M"
AGENT_NAME = "ollama_public_api_probe"

toolkit.show_json(
    {
        "package": toolkit.__name__,
        "version": toolkit.__version__,
        "run_live": RUN_OLLAMA_LIVE,
        "environment": ollama_environment,
    },
    title="Preflight Ollama",
)


## 1) Declarar runtime y límites

El endpoint por defecto apunta al servidor local oficial. Configurar una URL no
inicia el servidor ni garantiza que el modelo exista.


In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=120,
    max_retries=0,
    max_tool_calls=1,
    max_turns=3,
    max_concurrency=1,
)

runtime = toolkit.runtime(
    provider="ollama-runtime",
    model=MODEL,
    scheduler=scheduler,
    metadata={"tutorial": "providers/ollama"},
)

toolkit.show_json(runtime.describe(), title="Ollama RuntimeConfig")


## 2) Crear system, tool y agent

La Tool y el Agent no importan el SDK de Ollama. El provider conserva la
identidad `ollama-runtime` aunque use transporte OpenAI-compatible.


In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    '''Verifica un símbolo contra la superficie pública instalada.'''
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

system = toolkit.system(runtime=runtime)
agent = system.agent(
    name=AGENT_NAME,
    instructions=(
        "Usa inspect_public_api para verificar el símbolo solicitado. "
        "Responde con el nombre, si es público y la versión observada."
    ),
    tools=[inspect_public_api],
    contract=toolkit.AgentContract(
        must_call=["inspect_public_api"],
        completion="when_required_tools_satisfied",
    ),
    policy=toolkit.RunPolicy(
        max_turns=3,
        max_tool_calls=1,
        temperature=0.0,
        trace="compact",
        strict=True,
    ),
)

toolkit.show_json(agent.info(), title="Agente declarado")


## 3) Ejecutar o reportar not-run

La ejecución live requiere una señal explícita de Ollama. Usa
`RUN_OLLAMA_LIVE=0` para conservar un tutorial offline reproducible.


In [ ]:
can_run = RUN_OLLAMA_LIVE and bool(
    os.getenv("OLLAMA_MODEL") or os.getenv("OLLAMA_BASE_URL")
)

if can_run:
    result = agent.run(
        "Verifica si system pertenece a la API pública instalada.",
        mode="eval",
    )
    assert isinstance(result, toolkit.RunResult)
    assert result.ok, result.errors
    assert result.engine == "ollama-runtime"
    toolkit.human_result(result, title="Ollama RunResult", show_lineage=True)
    toolkit.show_json(
        toolkit.run_result_output(result),
        title="Contrato normalizado",
    )
else:
    result = None
    toolkit.show_json(
        {
            "status": "not-run",
            "provider": "ollama-runtime",
            "reason": (
                "Configura OLLAMA_MODEL u OLLAMA_BASE_URL, "
                "o usa RUN_OLLAMA_LIVE=0."
            ),
        },
        title="Ollama live gate",
    )


## 4) API realmente ejercitada

El preflight, la declaración y la ejecución usan exclusivamente la superficie
pública.


In [ ]:
api_coverage = [
    "toolkit.ollama_environment_snapshot",
    "toolkit.scheduler",
    "toolkit.runtime",
    "toolkit.tool",
    "toolkit.system",
    "system.agent",
    "agent.run",
    "toolkit.human_result",
    "toolkit.run_result_output",
    "toolkit.RunResult",
    "toolkit.show_json",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
]

toolkit.show_json(api_coverage, title="Ollama API coverage")


## Resultado e interpretacion

`not-run` prueba que el tutorial es seguro sin infraestructura. Sólo el bloque
live exitoso demuestra servidor, modelo, tool calling y normalización
`RunResult` de Ollama.
